# 04 数据获取、字段标准化与缓存

## 4.1 本章要解决什么问题

量化流程进入实战后，第一件事不是写策略，而是把真实行情稳定地取回来、整理成统一字段，并缓存成本地文件。  
本章用 AKShare 获取 ETF 日线数据，重点放在三件事：

- 把 AKShare 原始中文字段规范成 `date/code/open/high/low/close/volume/amount`。
- 说明日期参数、复权参数 `adjust` 和价格口径的常见坑。
- 使用缓存优先的方式读取真实数据：优先使用本地缓存，网络可用时再尝试 AKShare。

## 4.2 位置、输入与输出

- 上一章：已经掌握 pandas / numpy 处理表格与时间序列的基本操作。
- 本章输入：ETF 代码、日期区间、AKShare 接口、已有的 `data/sample/` 缓存。
- 本章输出：理解并验证 `data/sample/prices.parquet`、`assets.parquet`、`calendar.parquet` 的来源与结构。
- 下一章：直接读取标准化后的缓存，做清洗、对齐和收益率计算。

本 notebook 默认不刷新、不覆盖 `data/sample/`。实践性的联网抓取会写到 `outputs/results/ch04_data_cache/`，这样读者可以练习缓存流程，同时保留项目随附样例数据。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("请从 pyquant-roadmap 项目目录或 notebooks 目录运行本 notebook")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from lib.data.cache import load_parquet, save_parquet
from lib.data.sample import load_sample_assets, load_sample_calendar, load_sample_prices
from lib.data.schema import OHLCV_COLUMNS, normalize_ohlcv
from lib.paths import CONFIG_DIR, RESULTS_DIR, SAMPLE_DIR


def rel_path(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT).as_posix()


pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

CHAPTER_CACHE_DIR = RESULTS_DIR / "ch04_data_cache"
CHAPTER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "prices": SAMPLE_DIR / "prices.parquet",
    "assets": SAMPLE_DIR / "assets.parquet",
    "calendar": SAMPLE_DIR / "calendar.parquet",
}

print("project root: .")
print(f"sample cache: {rel_path(SAMPLE_DIR)}")
print(f"chapter scratch cache: {rel_path(CHAPTER_CACHE_DIR)}")


project root: .
sample cache: data/sample
chapter scratch cache: outputs/results/ch04_data_cache


## 4.3 学习路线

1. 先用一个手写小表理解“字段标准化”。
2. 再把同样规则交给 `lib.data.schema.normalize_ohlcv`。
3. 检查项目随附的 `data/sample/`，确认它是标准化后的真实 ETF 缓存。
4. 用 `lib.data.sources.fetch_akshare_etf_daily` 尝试联网抓取；如果 AKShare 或网络不可用，就退回本地缓存继续学习。
5. 最后明确什么时候可以刷新 `data/sample/`，以及为什么默认不在 notebook 中覆盖它。


## 4.4 第一层：手写最小字段标准化

AKShare 的 ETF 日线接口返回中文字段。后续清洗、因子、组合和回测章节不应该依赖任何单一数据源的原始字段名，所以本项目统一使用：

`date/code/open/high/low/close/volume/amount`

下面先不用网络，手写一张很小的“原始表”，模拟 AKShare 的字段形态。


In [2]:
raw_tiny = pd.DataFrame(
    {
        "日期": ["2023-01-04", "2023-01-03", "2023-01-03"],
        "开盘": ["4.02", "4.00", "4.00"],
        "最高": ["4.08", "4.05", "4.05"],
        "最低": ["4.01", "3.98", "3.98"],
        "收盘": ["4.06", "4.03", "4.03"],
        "成交量": ["1200000", "980000", "980000"],
        "成交额": ["4860000.0", "3949400.0", "3949400.0"],
    }
)

AKSHARE_ETF_COLUMN_MAP = {
    "日期": "date",
    "开盘": "open",
    "最高": "high",
    "最低": "low",
    "收盘": "close",
    "成交量": "volume",
    "成交额": "amount",
}


def normalize_ohlcv_manual(raw: pd.DataFrame, code: str) -> pd.DataFrame:
    out = raw.rename(columns=AKSHARE_ETF_COLUMN_MAP).assign(code=code).copy()

    missing = [col for col in OHLCV_COLUMNS if col not in out.columns]
    if missing:
        raise ValueError(f"missing OHLCV columns: {missing}")

    out = out[OHLCV_COLUMNS].copy()
    out["date"] = pd.to_datetime(out["date"])
    out["code"] = out["code"].astype(str)
    for col in ["open", "high", "low", "close", "volume", "amount"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.drop_duplicates(["date", "code"]).sort_values(["date", "code"]).reset_index(drop=True)


manual_prices = normalize_ohlcv_manual(raw_tiny, code="510300")

display(raw_tiny)
display(manual_prices)
print(manual_prices.dtypes)


,日期,开盘,最高,最低,收盘,成交量,成交额
0,2023-01-04,4.02,4.08,4.01,4.06,1200000,4860000.0
1,2023-01-03,4.00,4.05,3.98,4.03,980000,3949400.0
2,2023-01-03,4.00,4.05,3.98,4.03,980000,3949400.0


,date,code,open,high,low,close,volume,amount
0,2023-01-03,510300,4.00,4.05,3.98,4.03,980000,3949400.0
1,2023-01-04,510300,4.02,4.08,4.01,4.06,1200000,4860000.0


date      datetime64[ns]
code              object
open             float64
high             float64
low              float64
close            float64
volume             int64
amount           float64
dtype: object


这个小函数完成了数据接入时最核心的动作：

- 字段改名：把数据源字段翻译成项目字段。
- 日期解析：把字符串转成 pandas 的 `datetime64[ns]`。
- 数值解析：把价格、成交量、成交额转成可计算的数值列。
- 去重和排序：保证同一只标的同一天只有一条记录。

真实项目还会有停牌、缺失值、复权口径、数据源变更等问题，但字段标准化是所有后续处理的入口。


## 4.5 第二层：用 `lib` 复用同一套规则

手写版本是为了理解原则。项目代码里不要到处复制这段逻辑，而是通过 `lib.data.schema.normalize_ohlcv` 复用。  
它接受一个 `column_map`，所以同一个函数可以适配 AKShare，也可以适配其他数据源。


In [3]:
lib_prices = normalize_ohlcv(
    raw_tiny.assign(code="510300"),
    column_map=AKSHARE_ETF_COLUMN_MAP,
)

display(lib_prices)
assert lib_prices.equals(manual_prices)
print("lib.data.schema.normalize_ohlcv 与手写版本结果一致。")


,date,code,open,high,low,close,volume,amount
0,2023-01-03,510300,4.00,4.05,3.98,4.03,980000,3949400.0
1,2023-01-04,510300,4.02,4.08,4.01,4.06,1200000,4860000.0


lib.data.schema.normalize_ohlcv 与手写版本结果一致。


## 4.6 日期和复权参数的注意事项

AKShare 的 `fund_etf_hist_em` 接口常用参数包括：

- `symbol`：ETF 代码，例如 `510300`。
- `period="daily"`：日线。
- `start_date` / `end_date`：AKShare 接口通常需要 `YYYYMMDD`，本项目的 lib 包装函数允许读者传 `YYYY-MM-DD`。
- `adjust`：复权口径。`qfq` 通常表示前复权，适合本项目里做连续价格研究；空字符串通常表示不复权；`hfq` 通常表示后复权。

复权价格适合研究收益率和策略曲线，但不是当天真实可成交价格。做订单、滑点、成交额和实盘复盘时，要清楚自己使用的是哪种价格口径。


In [4]:
date_examples = pd.Series(["2023-01-01", "20230131", pd.Timestamp("2023-02-15")], name="input")

parsed_dates = date_examples.map(lambda value: pd.Timestamp(value))

date_table = pd.DataFrame(
    {
        "input": date_examples.astype(str),
        "pandas_timestamp": parsed_dates.dt.strftime("%Y-%m-%d"),
        "akshare_argument": parsed_dates.dt.strftime("%Y%m%d"),
    }
)

display(date_table)


,input,pandas_timestamp,akshare_argument
0,2023-01-01,2023-01-01,20230101
1,20230131,2023-01-31,20230131
2,2023-02-15 00:00:00,2023-02-15,20230215


## 4.7 `data/sample/`：项目随附的真实 ETF 缓存

为了让后续章节不依赖每次联网，项目仓库随附了一个小型真实 ETF 数据集。它已经按本章的标准字段保存为 parquet：

- `data/sample/prices.parquet`：标准化 OHLCV 行情。
- `data/sample/assets.parquet`：标的元数据。
- `data/sample/calendar.parquet`：从行情交易日提取的交易日历。

本章会读取并检查这些文件，但不会自动覆盖它们。


In [5]:
sample_assets = load_sample_assets()
sample_prices = load_sample_prices()
sample_calendar = load_sample_calendar()

path_table = pd.DataFrame(
    [
        {
            "name": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        }
        for name, path in SAMPLE_PATHS.items()
    ]
)

coverage = (
    sample_prices.groupby("code")["date"]
    .agg(start="min", end="max", rows="count")
    .reset_index()
    .merge(sample_assets[["code", "name"]], on="code", how="left")
    [["code", "name", "start", "end", "rows"]]
)

display(path_table)
display(sample_assets)
display(coverage)
display(sample_prices.head())


,name,path,exists,size_kb
0,prices,data\sample\prices.parquet,True,104.4
1,assets,data\sample\assets.parquet,True,2.7
2,calendar,data\sample\calendar.parquet,True,7.8


,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,code,name,start,end,rows
0,159915,创业板ETF,2021-01-04,2023-12-29,725
1,510300,沪深300ETF,2021-01-04,2023-12-29,725
2,510500,中证500ETF,2021-01-04,2023-12-29,725
3,512100,中证1000ETF,2021-01-04,2023-12-29,725


,date,code,open,high,low,close,volume,amount
0,2021-01-04,159915,2.864,2.990,2.861,2.976,2472764,7.278150e+08
1,2021-01-04,510300,4.789,4.875,4.769,4.843,5067056,2.697193e+09
2,2021-01-04,510500,5.926,6.046,5.899,6.014,3035862,2.166750e+09
3,2021-01-04,512100,2.493,2.545,2.488,2.537,1121520,1.064703e+08
4,2021-01-05,159915,2.940,2.997,2.925,2.988,2470004,7.346481e+08


## 4.8 缓存优先：联网抓取一只 ETF，失败就退回本地样例

下面的代码使用 `lib.data.sources.fetch_akshare_etf_daily` 调 AKShare。为了让 notebook 在离线、AKShare 限流或接口临时变更时仍能跑通，它采用缓存优先策略：

1. 如果 `outputs/results/ch04_data_cache/510300_2023-01_qfq.parquet` 已存在，直接读取。
2. 如果本章缓存不存在，尝试联网调用 AKShare，并把结果写入本章 scratch cache。
3. 如果联网失败，从 `data/sample/prices.parquet` 切出同一标的、同一日期区间作为后备数据。

这样可以展示真实抓取路径，同时不把网络可用性变成学习门槛。


In [6]:
from lib.data.sources import fetch_akshare_etf_daily

LIVE_SYMBOL = "510300"
LIVE_START = "2023-01-01"
LIVE_END = "2023-01-31"
LIVE_ADJUST = "qfq"
LIVE_CACHE_PATH = CHAPTER_CACHE_DIR / f"{LIVE_SYMBOL}_2023-01_{LIVE_ADJUST}.parquet"


def load_or_fetch_one_etf_cache_first(
    symbol: str,
    start_date: str,
    end_date: str,
    adjust: str,
    cache_path: Path,
) -> tuple[pd.DataFrame, str]:
    if cache_path.exists():
        return load_parquet(cache_path), "chapter_cache"

    try:
        fetched = fetch_akshare_etf_daily(symbol, start_date, end_date, adjust=adjust)
        save_parquet(fetched, cache_path)
        return fetched, "akshare_live_saved_to_chapter_cache"
    except Exception as exc:
        start = pd.Timestamp(start_date)
        end = pd.Timestamp(end_date)
        fallback = sample_prices[
            (sample_prices["code"] == symbol)
            & (sample_prices["date"].between(start, end))
        ].copy()
        if fallback.empty:
            raise RuntimeError(
                f"AKShare fetch failed and sample cache has no fallback rows for {symbol} "
                f"between {start_date} and {end_date}."
            ) from exc
        reason = f"sample_fallback_after_{type(exc).__name__}: {exc}"
        return fallback.reset_index(drop=True), reason


one_etf, data_source = load_or_fetch_one_etf_cache_first(
    LIVE_SYMBOL,
    LIVE_START,
    LIVE_END,
    LIVE_ADJUST,
    LIVE_CACHE_PATH,
)

display(one_etf.head())
print(f"data source: {data_source}")
print(f"chapter cache path: {LIVE_CACHE_PATH.relative_to(PROJECT_ROOT)}")
print(f"rows: {len(one_etf)}, date range: {one_etf['date'].min().date()} -> {one_etf['date'].max().date()}")
assert list(one_etf.columns) == OHLCV_COLUMNS


,date,code,open,high,low,close,volume,amount
0,2023-01-03,510300,3.593,3.617,3.546,3.605,6563108,2.576088e+09
1,2023-01-04,510300,3.604,3.621,3.587,3.610,9807997,3.874438e+09
2,2023-01-05,510300,3.631,3.691,3.628,3.685,7745023,3.108571e+09
3,2023-01-06,510300,3.686,3.718,3.682,3.701,5410808,2.187353e+09
4,2023-01-09,510300,3.721,3.742,3.708,3.726,7809599,3.178181e+09


data source: chapter_cache
chapter cache path: outputs\results\ch04_data_cache\510300_2023-01_qfq.parquet
rows: 16, date range: 2023-01-03 -> 2023-01-31


## 4.9 ETF 池配置和 `data/sample/` 刷新入口

项目主线不是只研究一只 ETF，而是研究一个小型 ETF 池。默认配置在 `configs/data_sources.yml`。  
真正刷新样例数据时，`lib.data.sources.fetch_and_save_akshare_etf_dataset` 会：

1. 逐只 ETF 调用 AKShare。
2. 标准化成统一 OHLCV 字段。
3. 只保留 ETF 池共同拥有的交易日，便于后续章节转成矩阵。
4. 生成 `assets.parquet`、`calendar.parquet`、`prices.parquet`。

为了保留仓库随附样例数据，下面的刷新开关默认关闭。


In [7]:
import yaml

from lib.data.sources import (
    EtfSymbol,
    build_asset_metadata,
    build_calendar_from_prices,
    fetch_and_save_akshare_etf_dataset,
)

cfg = yaml.safe_load((CONFIG_DIR / "data_sources.yml").read_text(encoding="utf-8"))["akshare_etf_universe"]
symbols = [EtfSymbol(item["code"], item["name"]) for item in cfg["symbols"]]

config_table = pd.DataFrame(
    {
        "code": [item.code for item in symbols],
        "name": [item.name for item in symbols],
        "start_date": cfg["start_date"],
        "end_date": cfg["end_date"],
        "adjust": cfg.get("adjust", "qfq"),
    }
)

display(config_table)
display(build_asset_metadata(symbols))
display(build_calendar_from_prices(sample_prices).head())

REFRESH_SAMPLE_CACHE = False

if REFRESH_SAMPLE_CACHE:
    refreshed = fetch_and_save_akshare_etf_dataset(
        symbols=symbols,
        start_date=cfg["start_date"],
        end_date=cfg["end_date"],
        adjust=cfg.get("adjust", "qfq"),
        output_dir=SAMPLE_DIR,
    )
    display(refreshed["paths"])
else:
    print("REFRESH_SAMPLE_CACHE=False: skip writing data/sample/.")
    print("To refresh intentionally, back up or review data/sample/ first, then set the flag to True.")


,code,name,start_date,end_date,adjust
0,510300,沪深300ETF,2021-01-01,2023-12-31,qfq
1,510500,中证500ETF,2021-01-01,2023-12-31,qfq
2,159915,创业板ETF,2021-01-01,2023-12-31,qfq
3,512100,中证1000ETF,2021-01-01,2023-12-31,qfq


,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,date,is_open
0,2021-01-04,1
1,2021-01-05,1
2,2021-01-06,1
3,2021-01-07,1
4,2021-01-08,1


REFRESH_SAMPLE_CACHE=False: skip writing data/sample/.
To refresh intentionally, back up or review data/sample/ first, then set the flag to True.


## 4.10 数据质量检查

获取和缓存只是第一步。把数据交给下一章之前，至少要检查：

- 字段是否完整且顺序一致。
- 日期是否已经转成 pandas datetime。
- 同一 `date/code` 是否只有一条记录。
- ETF 池在每个交易日的标的数量是否稳定。


In [8]:
assert list(sample_prices.columns) == OHLCV_COLUMNS
assert pd.api.types.is_datetime64_any_dtype(sample_prices["date"])
assert sample_prices.duplicated(["date", "code"]).sum() == 0

codes_per_day = sample_prices.groupby("date")["code"].nunique()
quality_summary = pd.DataFrame(
    {
        "metric": [
            "price_rows",
            "asset_count",
            "calendar_rows",
            "min_codes_per_day",
            "max_codes_per_day",
        ],
        "value": [
            len(sample_prices),
            sample_assets["code"].nunique(),
            len(sample_calendar),
            int(codes_per_day.min()),
            int(codes_per_day.max()),
        ],
    }
)

display(quality_summary)
display(sample_prices.groupby("code")[["open", "close", "volume", "amount"]].agg(["min", "max"]).round(4))


,metric,value
0,price_rows,2900
1,asset_count,4
2,calendar_rows,725
3,min_codes_per_day,4
4,max_codes_per_day,4


open         close          volume                 amount              
          min    max    min    max      min       max          min           max
code                                                                            
159915  1.749  3.455  1.755  3.455   643308  18945139  216778344.0  4.072251e+09
510300  3.075  5.504  3.089  5.388  1126992  28220118  561475936.0  1.096743e+10
510500  4.922  7.251  4.924  7.254   634108  11823844  472444970.0  6.900723e+09
512100  2.007  3.120  2.023  3.128   325107  26578040   29081660.0  3.179037e+09

## 4.11 小练习

把手写标准化函数应用到另一只 ETF 代码上，检查输出是否仍然满足项目字段顺序。  
真实使用时，只要数据源字段可以映射到标准字段，后续章节就不需要知道它来自哪里。


In [9]:
exercise_prices = normalize_ohlcv_manual(raw_tiny.iloc[:2], code="159915")

display(exercise_prices)
assert list(exercise_prices.columns) == OHLCV_COLUMNS
assert exercise_prices["code"].eq("159915").all()


,date,code,open,high,low,close,volume,amount
0,2023-01-03,159915,4.00,4.05,3.98,4.03,980000,3949400.0
1,2023-01-04,159915,4.02,4.08,4.01,4.06,1200000,4860000.0


## 4.12 本章产出与下一章衔接

现在我们已经有了标准化并可复用的行情入口：

- 原理层：知道如何从原始中文字段整理成标准 OHLCV。
- 实践层：会用 `lib.data.sources.fetch_akshare_etf_daily` 和 parquet 缓存做 cache-first 抓取。
- 项目层：确认 `data/sample/` 中的真实 ETF 缓存可以支撑后续章节。

第 05 章会把 `data/sample/prices.parquet`、`calendar.parquet` 和 `assets.parquet` 当作输入，继续处理缺失、对齐和收益率。
